# Stress Testing & Scenario Modeling

### Credit Scoring Risk Assessment — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), and basic understanding of credit scoring and risk assessment terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to credit scoring and risk assessment.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Credit scoring, probability of default, loss estimation, and stress testing for banking risk management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Banks must prove to regulators (like the Federal Reserve) that they can survive a major recession. This is called "Stress Testing." We use AI models to predict how individual loans react when unemployment rises and house prices fall.

**Input data used:** LTV, credit score, current unemployment rate, and house price index (HPI).

**What we predict:** Expected portfolio loss under different economic paths.

**Method used:** Monte Carlo-style simulation using a pre-trained risk model.

**Learning goal:** Learn how to use a predictive model to ask "What If?" questions about the future.

**Primary stakeholders:** credit analysts, loan officers, risk managers, and regulatory auditors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Portfolio & Risk Model

We'll create a simple function that represents our "Risk Model."

In [ ]:
n = 5000
portfolio = pd.DataFrame({
    'loan_id': range(n),
    'balance': RNG.gamma(10, 20000, n), # Loan amounts
    'ltv': RNG.uniform(0.6, 0.95, n),    # Loan-to-Value
    'credit_score': RNG.normal(700, 50, n).clip(300, 850)
})

def predict_default_prob(df, unemployment, hpi_change):
    # Logic: Risk increases if unemployment is high or house prices (HPI) drop
    # HPI drop increases effective LTV (house worth less than loan)
    adj_ltv = df['ltv'] * (1 - hpi_change)
    logit = (
        (800 - df['credit_score']) / 100 + 
        5.0 * adj_ltv + 
        20.0 * unemployment - 
        10.0
    )
    return 1 / (1 + np.exp(-logit))

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

| Scenario | Unemployment | House Price Change |
| --- | --- | --- |
| **Baseline** | 4% | +2% (Growth) |
| **Adverse** | 10% | -15% (Crash) |

In [ ]:
portfolio['pd_baseline'] = predict_default_prob(portfolio, unemployment=0.04, hpi_change=0.02)
portfolio['pd_adverse'] = predict_default_prob(portfolio, unemployment=0.10, hpi_change=-0.15)

expected_loss_baseline = (portfolio['pd_baseline'] * portfolio['balance']).sum()
expected_loss_adverse = (portfolio['pd_adverse'] * portfolio['balance']).sum()

print(f"Total Portfolio Balance: \${portfolio['balance'].sum()/1e6:.1f}M")
print(f"Expected Loss (Baseline): \${expected_loss_baseline/1e6:.2f}M")
print(f"Expected Loss (Adverse): \${expected_loss_adverse/1e6:.2f}M")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(portfolio['pd_baseline'], label='Baseline Scenario (Stable)', fill=True)
sns.kdeplot(portfolio['pd_adverse'], label='Adverse Scenario (Crisis)', fill=True)
plt.title('Shift in Loan Default Probabilities under Stress')
plt.xlabel('Probability of Default')
plt.ylabel('Density')
plt.legend()
plt.xlim(0, 0.5)
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Visualisation titled "Shift in Loan Default Probabilities under Stress" with Probability of Default on the x-axis and Density on the y-axis. The chart highlights key patterns that inform the modelling approach.

In [ ]:
unemployment_range = np.linspace(0.03, 0.15, 13)
losses = []

for u in unemployment_range:
    pd_series = predict_default_prob(portfolio, unemployment=u, hpi_change=0)
    losses.append((pd_series * portfolio['balance']).sum() / 1e6)

plt.plot(unemployment_range * 100, losses, 'o-')
plt.title('Sensitivity: Portfolio Loss vs. Unemployment Rate')
plt.xlabel('Unemployment Rate (%)')
plt.ylabel('Expected Loss ($ Millions)')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart titled "Sensitivity: Portfolio Loss vs. Unemployment Rate" with Unemployment Rate (%) on the x-axis and Expected Loss ($ Millions) on the y-axis. The chart shows temporal trends and the alignment between predicted and observed values.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary of Findings:**
- Under the **Adverse Scenario**, the expected loss nearly triples.
- The **Shift in Density** shows that many "Safe" loans become risky when the economy deteriorates.
- **Sensitivity Analysis** shows that once unemployment crosses 8%, the loss curve starts to steepen sharply.

**Banking Context:** These results are used to determine the **Capital Buffer**. If the adverse loss is \$50M, the bank must ensure it has at least that much extra capital to survive the "Stress Case" without failing.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end credit scoring and risk assessment workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Credit models can perpetuate historical discrimination if trained on biased lending data.
- Proxy variables such as zip code or employment type may correlate with protected characteristics.
- Applicants deserve transparent explanations of adverse credit decisions under fair lending regulations.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in credit scoring and risk assessment?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.